In [ ]:
# doc = Document(
#     page_content="This is the main text content to create RAG",
#     metadata={
#         "source": "example.txt",
#         "pages": 1,
#         "author": "Me",
#         "date" : "2025-12-5"
#     }
# )
# doc

In [ ]:
# import os
# os.makedirs("../data/text_files",exist_ok=True)

In [ ]:
# from langchain_community.document_loaders import TextLoader
# 
# loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
# document = loader.load()
# print(document)

In [ ]:
# from langchain_community.document_loaders import DirectoryLoader
# 
# dir_loader = DirectoryLoader(
#     "../data/text_files",
#     glob = "**/*.txt",
#     loader_cls = TextLoader,
#     loader_kwargs= {'encoding':"utf-8"},
#     show_progress=False
# )
# 
# docs = dir_loader.load()
# docs

In [ ]:
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents = process_all_pdfs("../data")

In [ ]:
### Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")     
    return split_docs
chunks=split_documents(all_pdf_documents)
for chunk in chunks:
    print (chunk.page_content)
    print("==============================================================================\n")
chunks

Embedding and vectorStoreDB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer #embedding model will be here
import chromadb
from chromadb.config import Settings
import uuid #every record we will put in vectordb will have an id
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #model that converts text into vectors
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model is loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model: {e}")
            raise

    def generate_embeddings(self, texts: List[str])-> np.ndarray:
        if not self.model:
            raise ValueError("No Model") 
        print(f"Generating Embeddings for {len(texts)}")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


### Vector store

In [ ]:
class VectorStore:
    def __init__(self,collection_name: str= "pdf_documents", persist_directory: str = "../data/vector_store"): #any generated vector will be stored in persistent directory
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path= self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"desciption" :"anything for now"}
            )

            print(f"Vector store initialized. collection : {self.collection_name}")
            print(f"collection currently has {self.collection.count()} docs")
        except Exception as e:
            print(f"Error happened: {e}")
            raise

    def add_docs(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents doesn't match the number of embeddings")

        ids=[]
        metadatas=[]
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in  enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids= ids,
                embeddings= embeddings_list,
                documents = documents_text,
                metadatas= metadatas
            )
            print(f"added {len(documents)} to vector store")

        except Exception as e:
            print(f"Error : {e}")
            raise

    def clear_collection(self):
        self.client.delete_collection(self.collection_name)
        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={"description": "anything for now"}
    )
    print(f"Collection {self.collection_name} cleared")

vectorStore = VectorStore()
vectorStore
    


In [ ]:
### convert the text into embeddings
texts = [doc.page_content for doc in chunks]

# generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)
print(f"Debug: embeddings shape {embeddings.shape}")

#store in the vector database
vectorStore.clear_collection()
vectorStore.add_docs(chunks,embeddings)

### Retriever from VecStore


In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float= 0.0) -> List[Dict[str,Any]]:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results= top_k
            )
            print(f"Debug: asked for {top_k}, got {len(results.get('documents', [[]])[0]) if results.get('documents') else 0} raw hits")

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - (distance / 2.0)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
            
            print(f"Debug: returning {len(retrieved_docs)} results")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorStore,embedding_manager)
rag_retriever
rag_retriever.retrieve("How well can LLMs generate CI configurations natural language descriptions?")